In [ ]:
!pip install -q kagglehub torch torchvision seaborn scikit-learn pandas matplotlib

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, accuracy_score
import kagglehub

sns.set_theme(style="whitegrid")
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
path = kagglehub.dataset_download("tsaideepak/skin-cancer")
base = Path(path) / "datatree"

train_dir = base / "train"
val_dir = base / "validation"

print("base =", base)

In [ ]:
def count_split(d):
    return {cls: len(list((d/cls).glob('*'))) for cls in sorted(p.name for p in d.iterdir() if p.is_dir())}

train_counts = count_split(train_dir)
val_counts = count_split(val_dir)

df_counts = pd.DataFrame({'train':train_counts,'validation':val_counts}).fillna(0).astype(int)
display(df_counts)

plot_df = df_counts.reset_index().melt(id_vars='index', var_name='split', value_name='count')

plt.figure(figsize=(10,4))
sns.barplot(data=plot_df, x='index', y='count', hue='split')
plt.title('Class imbalance profile')
plt.show()

In [ ]:
tfm_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

tfm_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
ds_train = datasets.ImageFolder(train_dir, transform=tfm_train)
ds_val = datasets.ImageFolder(val_dir, transform=tfm_val)

classes = ds_train.classes

loader_tr = DataLoader(ds_train, batch_size=32, shuffle=True)
loader_va = DataLoader(ds_val, batch_size=64, shuffle=False)

In [ ]:
from collections import Counter

# Calculate weights for class imbalance
targets = ds_train.targets
counts = Counter(targets)

total = sum(counts.values())
weights = [total/counts[i] for i in range(len(counts))]
weights = torch.tensor(weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

In [ ]:
import torchvision.models as models
import torch.nn as nn

# Using EfficientNet_B0 for better performance and matching the backend
model = models.efficientnet_b0(pretrained=True)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))

model = model.to(device)

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=15)

In [ ]:
# -----------------------------
# STAGE 1: FREEZE + TRAIN HEAD
# -----------------------------
for p in model.parameters():
    p.requires_grad = False

for p in model.classifier[1].parameters():
    p.requires_grad = True

epochs = 5
for ep in range(epochs):
    model.train()
    run = 0.0
    for x, y in loader_tr:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        opt.step()
        run += loss.item()
    print(f'[Stage1] epoch {ep+1}/{epochs} loss={run/len(loader_tr):.4f}')

In [ ]:
# -----------------------------
# STAGE 2: UNFREEZE ALL + FINE-TUNE
# -----------------------------
for p in model.parameters():
    p.requires_grad = True

opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)

epochs = 20
for ep in range(epochs):
    model.train()
    run = 0.0
    for x,y in loader_tr:
        x,y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        opt.step()
        run += loss.item()
    scheduler.step()
    print(f'[Stage2] epoch {ep+1}/{epochs} loss={run/len(loader_tr):.4f}')

In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for x, y in loader_va:
        x = x.to(device)
        outputs = model(x)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)
        y_pred.extend(preds.cpu().numpy())
        y_true.extend(y.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average='macro', zero_division=0))
print("Recall   :", recall_score(y_true, y_pred, average='macro', zero_division=0))
print("F1 Score :", f1_score(y_true, y_pred, average='macro'))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Save and Download
torch.save(model.state_dict(), "model.pth")
print("Model saved!")

try:
    from google.colab import files
    files.download('model.pth')
except:
    print("Not in Colab, manual download needed.")